In [ ]:
from pathlib import Path
import json
import shutil

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

In [ ]:
BASE_DIR = Path.cwd().parent.resolve()  # Ejecutar desde pec3/notebooks

DATA_DIR = BASE_DIR / "data"

SMART_DIR = DATA_DIR / "SMART-OM"
COLAB_DIR = DATA_DIR / "imgsMIIA_colab"

IMGSMIIA_PREP_DIR = COLAB_DIR / "prepared"

FINAL_DIR = DATA_DIR / "oral_seg_final"

FINAL_IMAGES_DIR = FINAL_DIR / "images"
FINAL_MASKS_DIR = FINAL_DIR / "masks"

OUTPUT_DIR = BASE_DIR / "outputs"
TABLES_DIR = OUTPUT_DIR / "tables"
FIGURES_DIR = OUTPUT_DIR / "figures"

for carpeta in [FINAL_IMAGES_DIR, FINAL_MASKS_DIR, TABLES_DIR, FIGURES_DIR]:
    carpeta.mkdir(parents=True, exist_ok=True)

SMART_CSV = TABLES_DIR / "smart_om_region_annotations_validas.csv"
IMGSMIIA_CSV = TABLES_DIR / "imgsMIIA_masks_validas.csv"

print("SMART_CSV existe:", SMART_CSV.exists())
print("IMGSMIIA_CSV existe:", IMGSMIIA_CSV.exists())
print("FINAL_DIR:", FINAL_DIR)

In [ ]:
if FINAL_DIR.exists():
    shutil.rmtree(FINAL_DIR)

for split in ["train", "val", "test"]:
    (FINAL_IMAGES_DIR / split).mkdir(parents=True, exist_ok=True)
    (FINAL_MASKS_DIR / split).mkdir(parents=True, exist_ok=True)

print("Dataset final limpio.")

In [ ]:
IMG_SIZE = 256
RANDOM_STATE = 7

print("Tamaño final:", IMG_SIZE, "x", IMG_SIZE)

In [ ]:
def detectar_clase_lengua(texto):
    texto = str(texto).upper()
    
    if "_DT" in texto or "- DT" in texto or " DT" in texto or "DORSAL" in texto:
        return "DT"
    
    if "_VT" in texto or "- VT" in texto or " VT" in texto or "VENTRAL" in texto:
        return "VT"
    
    if "_LT" in texto or "- LT" in texto or " LT" in texto or "LATERAL" in texto:
        return "LT"
    
    return "desconocida"

In [ ]:
df_smart = pd.read_csv(SMART_CSV)
df_imgsm = pd.read_csv(IMGSMIIA_CSV)

print("SMART-OM anotaciones:", len(df_smart))
print("imgsMIIA anotaciones:", len(df_imgsm))

display(df_smart.head())
display(df_imgsm.head())

In [ ]:
clases_smart = []

for i, fila in df_smart.iterrows():
    texto = (
        str(fila.get("filename", "")) + " " +
        str(fila.get("img_relativa", "")) + " " +
        str(fila.get("json_relativo", ""))
    )
    
    clase = detectar_clase_lengua(texto)
    clases_smart.append(clase)

df_smart["clase_lengua"] = clases_smart

print("Distribución SMART antes de filtrar:")
print(df_smart["clase_lengua"].value_counts())

df_smart_lengua = df_smart[df_smart["clase_lengua"] != "desconocida"].copy()

print("\nSMART-OM lengua:", len(df_smart_lengua))
print(df_smart_lengua["clase_lengua"].value_counts())

display(df_smart_lengua.head())

In [ ]:
def leer_json(json_path):
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    
    return data

In [ ]:
def limpiar_regions(regions):
    if regions is None:
        return []
    
    if isinstance(regions, dict):
        regions = list(regions.values())
    
    if not isinstance(regions, list):
        return []
    
    regions_limpias = []
    
    for region in regions:
        if region is None:
            continue
        
        if not isinstance(region, dict):
            continue
        
        if "shape_attributes" not in region:
            continue
        
        regions_limpias.append(region)
    
    return regions_limpias

In [ ]:
def buscar_regions_en_json(json_path, filename_buscado):
    """
    Busca dentro del JSON VIA la entrada de una imagen concreta.
    Devuelve sus regions.
    """
    
    data = leer_json(json_path)
    
    if "_via_img_metadata" not in data:
        return []
    
    metadata = data["_via_img_metadata"]
    
    for img_id in metadata:
        item = metadata[img_id]
        
        filename = item.get("filename", None)
        
        if filename == filename_buscado:
            regions = item.get("regions", [])
            
            if isinstance(regions, dict):
                regions = list(regions.values())
            
            return regions
    
    return []

In [ ]:
def crear_mascara_desde_regions(regions, alto, ancho):
    mask = np.zeros((alto, ancho), dtype=np.uint8)
    
    regions = limpiar_regions(regions)
    
    for region in regions:
        shape = region.get("shape_attributes", {})
        nombre = shape.get("name", "")
        
        if nombre == "polygon":
            xs = shape.get("all_points_x", [])
            ys = shape.get("all_points_y", [])
            
            if len(xs) >= 3 and len(xs) == len(ys):
                puntos = np.array(list(zip(xs, ys)), dtype=np.int32)
                
                puntos[:, 0] = np.clip(puntos[:, 0], 0, ancho - 1)
                puntos[:, 1] = np.clip(puntos[:, 1], 0, alto - 1)
                
                puntos = puntos.reshape((-1, 1, 2))
                cv2.fillPoly(mask, [puntos], 255)
        
        elif nombre == "rect":
            x = int(shape.get("x", 0))
            y = int(shape.get("y", 0))
            w = int(shape.get("width", 0))
            h = int(shape.get("height", 0))
            
            x1 = max(0, x)
            y1 = max(0, y)
            x2 = min(ancho - 1, x + w)
            y2 = min(alto - 1, y + h)
            
            if x2 > x1 and y2 > y1:
                cv2.rectangle(mask, (x1, y1), (x2, y2), 255, -1)
        
        elif nombre == "circle":
            cx = int(shape.get("cx", 0))
            cy = int(shape.get("cy", 0))
            r = int(shape.get("r", 0))
            
            if r > 0:
                cv2.circle(mask, (cx, cy), r, 255, -1)
        
        elif nombre == "ellipse":
            cx = int(shape.get("cx", 0))
            cy = int(shape.get("cy", 0))
            rx = int(shape.get("rx", 0))
            ry = int(shape.get("ry", 0))
            
            if rx > 0 and ry > 0:
                cv2.ellipse(mask, (cx, cy), (rx, ry), 0, 0, 360, 255, -1)
    
    return mask

In [ ]:
filas_smart = []
errores_smart = []

contador = 0

for i, fila in df_smart_lengua.iterrows():
    img_path = Path(fila["img_path"])
    json_path = Path(fila["json_path"])
    filename = fila["filename"]
    clase_lengua = fila["clase_lengua"]
    
    img = cv2.imread(str(img_path), cv2.IMREAD_COLOR)
    
    if img is None:
        errores_smart.append({
            "origen": "SMART-OM",
            "filename": filename,
            "motivo": "opencv_no_lee_imagen"
        })
        continue
    
    alto, ancho = img.shape[:2]
    
    regions = buscar_regions_en_json(json_path, filename)
    
    mask = crear_mascara_desde_regions(regions, alto, ancho)
    
    if mask.max() == 0:
        errores_smart.append({
            "origen": "SMART-OM",
            "filename": filename,
            "motivo": "mascara_vacia"
        })
        continue
    
    filas_smart.append({
        "origen": "SMART-OM",
        "clase_lengua": clase_lengua,
        "img_path": str(img_path),
        "mask_path": "",
        "json_path": str(json_path),
        "filename_original": filename
    })
    
    contador += 1

df_smart_final = pd.DataFrame(filas_smart)
df_errores_smart = pd.DataFrame(errores_smart)

print("SMART-OM lengua válido:", len(df_smart_final))
print("Errores SMART:", len(df_errores_smart))

if len(df_smart_final) > 0:
    print(df_smart_final["clase_lengua"].value_counts())
    display(df_smart_final.head())

if len(df_errores_smart) > 0:
    display(df_errores_smart.head())

In [ ]:
filas_imgsm = []

for i, fila in df_imgsm.iterrows():
    filas_imgsm.append({
        "origen": "imgsMIIA_colab",
        "clase_lengua": fila["clase_lengua"],
        "img_path": fila["img_preparada"],
        "mask_path": fila["mask_preparada"],
        "json_path": fila["json_path"],
        "filename_original": fila["filename_original"]
    })

df_imgsm_final = pd.DataFrame(filas_imgsm)

print("imgsMIIA válido:", len(df_imgsm_final))
print(df_imgsm_final["clase_lengua"].value_counts())

display(df_imgsm_final.head())

In [ ]:
df_total = pd.concat(
    [df_smart_final, df_imgsm_final],
    ignore_index=True
)

df_total["id_global"] = np.arange(len(df_total))

print("Total dataset combinado:", len(df_total))
print("\nPor origen:")
print(df_total["origen"].value_counts())

print("\nPor clase:")
print(df_total["clase_lengua"].value_counts())

print("\nCruzado:")
display(pd.crosstab(df_total["origen"], df_total["clase_lengua"]))

display(df_total.head())

In [ ]:
df_total["estrato"] = df_total["origen"] + "_" + df_total["clase_lengua"]

conteos = df_total["estrato"].value_counts()
print(conteos)

usar_stratify = True

if conteos.min() < 2:
    usar_stratify = False

if usar_stratify:
    strat_1 = df_total["estrato"]
else:
    strat_1 = None

df_train, df_temp = train_test_split(
    df_total,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=strat_1
)

if usar_stratify:
    strat_2 = df_temp["estrato"]
else:
    strat_2 = None

df_val, df_test = train_test_split(
    df_temp,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=strat_2
)

df_train = df_train.copy()
df_val = df_val.copy()
df_test = df_test.copy()

df_train["split"] = "train"
df_val["split"] = "val"
df_test["split"] = "test"

df_split = pd.concat([df_train, df_val, df_test], ignore_index=True)

print("Train:", len(df_train))
print("Val:", len(df_val))
print("Test:", len(df_test))

print("\nCruzado por split/origen:")
display(pd.crosstab(df_split["split"], df_split["origen"]))

print("\nCruzado por split/clase:")
display(pd.crosstab(df_split["split"], df_split["clase_lengua"]))

In [ ]:
def guardar_par_final(img_path, mask_path, json_path, filename_original, origen, clase_lengua, id_global, split):
    """
    Guarda imagen y máscara redimensionadas a IMG_SIZE.
    Para SMART-OM genera la máscara desde JSON.
    Para imgsMIIA usa la máscara ya preparada.
    """
    
    img_path = Path(img_path)
    img = cv2.imread(str(img_path), cv2.IMREAD_COLOR)
    
    if img is None:
        return None
    
    alto, ancho = img.shape[:2]
    
    if origen == "SMART-OM":
        json_path = Path(json_path)
        regions = buscar_regions_en_json(json_path, filename_original)
        mask = crear_mascara_desde_regions(regions, alto, ancho)
    else:
        mask_path = Path(mask_path)
        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    
    if mask is None:
        return None
    
    if mask.max() == 0:
        return None
    
    img_res = cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
    mask_res = cv2.resize(mask, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)
    
    mask_res = (mask_res > 0).astype(np.uint8) * 255
    
    nombre = f"{int(id_global):05d}_{origen}_{clase_lengua}"
    nombre = nombre.replace(" ", "_")
    
    img_out = FINAL_IMAGES_DIR / split / f"{nombre}.jpg"
    mask_out = FINAL_MASKS_DIR / split / f"{nombre}.png"
    
    cv2.imwrite(str(img_out), img_res)
    cv2.imwrite(str(mask_out), mask_res)
    
    return str(img_out), str(mask_out)

In [ ]:
filas_finales = []
errores_finales = []

for i, fila in df_split.iterrows():
    resultado = guardar_par_final(
        img_path=fila["img_path"],
        mask_path=fila["mask_path"],
        json_path=fila["json_path"],
        filename_original=fila["filename_original"],
        origen=fila["origen"],
        clase_lengua=fila["clase_lengua"],
        id_global=fila["id_global"],
        split=fila["split"]
    )
    
    if resultado is None:
        errores_finales.append({
            "id_global": fila["id_global"],
            "origen": fila["origen"],
            "clase_lengua": fila["clase_lengua"],
            "motivo": "no_se_pudo_guardar"
        })
        continue
    
    img_final, mask_final = resultado
    
    filas_finales.append({
        "id_global": fila["id_global"],
        "split": fila["split"],
        "origen": fila["origen"],
        "clase_lengua": fila["clase_lengua"],
        "img_final": img_final,
        "mask_final": mask_final,
        "filename_original": fila["filename_original"]
    })

df_dataset_final = pd.DataFrame(filas_finales)
df_errores_final = pd.DataFrame(errores_finales)

print("Pares finales guardados:", len(df_dataset_final))
print("Errores finales:", len(df_errores_final))

display(df_dataset_final.head())

if len(df_errores_final) > 0:
    display(df_errores_final.head())

In [ ]:
csv_final = TABLES_DIR / "oral_seg_dataset_final.csv"
csv_errores = TABLES_DIR / "oral_seg_dataset_errores.csv"

df_dataset_final.to_csv(csv_final, index=False)
df_errores_final.to_csv(csv_errores, index=False)

print("CSV final:", csv_final)
print("CSV errores:", csv_errores)

In [ ]:
print("Resumen por split:")
display(df_dataset_final["split"].value_counts())

print("Resumen por origen:")
display(df_dataset_final["origen"].value_counts())

print("Resumen por clase:")
display(df_dataset_final["clase_lengua"].value_counts())

print("Split vs origen:")
display(pd.crosstab(df_dataset_final["split"], df_dataset_final["origen"]))

print("Split vs clase:")
display(pd.crosstab(df_dataset_final["split"], df_dataset_final["clase_lengua"]))

In [ ]:
muestras = df_dataset_final.sample(
    n=min(8, len(df_dataset_final)),
    random_state=7
)

plt.figure(figsize=(12, 8))

for i, (_, fila) in enumerate(muestras.iterrows()):
    img = cv2.imread(fila["img_final"], cv2.IMREAD_COLOR)
    mask = cv2.imread(fila["mask_final"], cv2.IMREAD_GRAYSCALE)
    
    if img is None or mask is None:
        continue
    
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    plt.subplot(2, 4, i + 1)
    plt.imshow(img_rgb)
    plt.imshow(mask, cmap="jet", alpha=0.35)
    plt.title(fila["origen"] + " " + fila["clase_lengua"], fontsize=8)
    plt.axis("off")

plt.tight_layout()

fig_path = FIGURES_DIR / "oral_seg_dataset_final_muestras.png"
plt.savefig(fig_path, dpi=150)
plt.show()

print("Figura guardada:", fig_path)

In [ ]:
for split in ["train", "val", "test"]:
    n_img = len(list((FINAL_IMAGES_DIR / split).glob("*.jpg")))
    n_mask = len(list((FINAL_MASKS_DIR / split).glob("*.png")))
    
    print(split)
    print(" imágenes:", n_img)
    print(" máscaras:", n_mask)